# Gabriele's Notebook

In this notebook i test several models

In [5]:
import pandas as pd
import numpy as np

# Plots
import matplotlib.pyplot as plt
import seaborn as sns
from plotly import graph_objects as go

# Utils
from tqdm.notebook import tqdm
from tabulate import tabulate

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

# Custom modules
import Models_regr as LSTM
import Dataset
import utils


# Set SEED
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


## Data loading

We first load the dataframe from the parquet file.

In [6]:
df = pd.read_parquet('../Data/LOBSTER/AMZN.parquet')
df = df.head(20000)
target_cols = ['MP']

# Basic feature extraction
# df = utils.FeaturesExtractionLOB(df).extract()
df["MP"] = (df["P_Ask_1"] + df["P_Bid_1"]) / 2

df.head()

,P_Ask_1,V_Ask_1,P_Bid_1,V_Bid_1,P_Ask_2,V_Ask_2,P_Bid_2,V_Bid_2,P_Ask_3,V_Ask_3,...,V_Bid_8,P_Ask_9,V_Ask_9,P_Bid_9,V_Bid_9,P_Ask_10,V_Ask_10,P_Bid_10,V_Bid_10,MP
0,2239500,100,2231800,100,2239900,100,2230700,200,2240000,220,...,5000,2294300,100,2202000,100,2298000,100,2189700,100,2235650.0
1,2239500,100,2238100,21,2239900,100,2231800,100,2240000,220,...,100,2294300,100,2202500,5000,2298000,100,2202000,100,2238800.0
2,2239500,100,2238100,21,2239600,20,2231800,100,2239900,100,...,100,2267700,100,2202500,5000,2294300,100,2202000,100,2238800.0
3,2239500,100,2238100,21,2239600,20,2237500,100,2239900,100,...,4000,2267700,100,2204000,100,2294300,100,2202500,5000,2238800.0
4,2239500,100,2238100,21,2239600,20,2237500,100,2239900,100,...,4000,2267700,100,2204000,100,2294300,100,2202500,5000,2238800.0


In [7]:
# Split the dataframe into train and test
df_train, df_test = train_test_split(df, test_size=0.2, shuffle=False)
df_train, df_val = train_test_split(df_train, test_size=0.2, shuffle=False)

# Standardize the data
scaler = StandardScaler()
scaler.set_output()
df_train[df_train.columns] = scaler.fit_transform(df_train)
df_val[df_val.columns] = scaler.transform(df_val)
df_test[df_test.columns] = scaler.transform(df_test)

# Split covariates and response variable(s)
X_train, y_train = df_train.drop(columns=target_cols).values , df_train[target_cols].values
X_val, y_val     = df_val.drop(columns=target_cols).values   , df_val[target_cols].values
X_test, y_test   = df_test.drop(columns=target_cols).values  , df_test[target_cols].values

# Standardize the covariates
# scaler_X = StandardScaler()
# X_train = scaler_X.fit_transform(X_train)
# X_val = scaler_X.transform(X_val)
# X_test = scaler_X.transform(X_test)

# Build tensors 
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
X_val = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val, dtype=torch.float32).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

# Build TimeSeriesDataset
SEQ_LEN_X = 10
SEQ_LEN_Y = 1
OFFSET = 10
dataset_train = Dataset.TimeSeriesDataset(
    X         = X_train,
    y         = y_train,
    seq_len_x = SEQ_LEN_X,
    seq_len_y = SEQ_LEN_Y,
    offset    = OFFSET,
    channels  = True
)
dataset_val = Dataset.TimeSeriesDataset(
    X         = X_val,
    y         = y_val,
    seq_len_x = SEQ_LEN_X,
    seq_len_y = SEQ_LEN_Y,
    offset    = OFFSET,
    channels  = True
)
dataset_test = Dataset.TimeSeriesDataset(
    X         = X_test,
    y         = y_test,
    seq_len_x = SEQ_LEN_X,
    seq_len_y = SEQ_LEN_Y,
    offset    = OFFSET,
    channels  = True
)

# Build TimeSeriesLoader
BATCH_SIZE = 32
dataloader_train = Dataset.TimeSeriesLoader(
    dataset = dataset_train,
    batch_size = BATCH_SIZE,
    shuffle = True
)
dataloader_val = Dataset.TimeSeriesLoader(
    dataset = dataset_val,
    batch_size = BATCH_SIZE,
    shuffle = False
)
dataloader_test = Dataset.TimeSeriesLoader(
    dataset = dataset_test,
    batch_size = 1024,
    shuffle = False
)

In [8]:
print(f'Train dataset: {len(dataset_train)} samples')
print(f'Test dataset: {len(dataset_test)} samples')

Train dataset: 12780 samples
Test dataset: 3980 samples


## Models

### Aragosta

In [9]:
# Model parameters
INPUT_SIZE = X_train.shape[1]
OUTPUT_SIZE = y_train.shape[1]
HIDDEN_SIZE = 64

# LSTM
LSTM_SIZE = 64
LSTM_LAYERS = 2

# CNN
CONV_CHANNELS = 32
INCEPTION_CHANNELS = 64

# Optimizer parameters
LEARNING_RATE = 1e-3

# Build the model
model = LSTM.Aragosta(
    input_size = INPUT_SIZE,
    output_size = OUTPUT_SIZE,
    hidden_size = HIDDEN_SIZE,
    lstm_size = LSTM_SIZE,
    lstm_layers = LSTM_LAYERS,
    conv_channels = CONV_CHANNELS,
    inception_channels = INCEPTION_CHANNELS,
).to(device)

# Print the number of parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'The model has {n_params} parameters')

# Loss function
criterion = torch.nn.MSELoss()
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Train the model
train_res = utils.train(
    train_loader = dataloader_train,
    val_loader = dataloader_val,
    model = model,
    optimizer = optimizer,
    criterion = criterion,
    device = device,
    n_epochs = 3,
)

# Plot loss evolution
plt.figure(figsize=(10, 5))
plt.plot(train_res['losses_train'], label='Train loss')
plt.plot(train_res['losses_val'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

The model has 211681 parameters


Training:   0%|          | 0/3 [00:00<?, ?it/s]

c:\Users\gabri\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\rnn.py:879: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at ..\aten\src\ATen\native\cudnn\RNN.cpp:982.)
  result = _VF.lstm(input, hx, self._flat_weights, self.bias, self.num_layers,


RuntimeError: shape '[32768, 1]' is invalid for input of size 65536

In [ ]:
evaluation_res = utils.evaluate(
    model = model,
    loader = dataloader_test,
    criterion = criterion,
    device = device
)

true = np.concatenate(evaluation_res['true'], axis=0)
pred = np.concatenate(evaluation_res['pred'], axis=0)

print(f'sMAPE: {utils.sMAPE(true,pred):.1f}%')

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(true)), y=true, mode='lines', name='True'))
fig.add_trace(go.Scatter(x=np.arange(len(pred)), y=pred, mode='lines', name='Predicted'))
fig.show()

sMAPE: 19.6%


### Basic LSTM

In [ ]:
# Model parameters
INPUT_SIZE = X_train.shape[1]
OUTPUT_SIZE = y_train.shape[1]
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.0
# Optimizer parameters
LEARNING_RATE = 1e-3

# Build the model
model = LSTM.LSTM(
    input_size = INPUT_SIZE,
    output_size = OUTPUT_SIZE,
    hidden_size = HIDDEN_SIZE,
    num_layers = NUM_LAYERS,
    dropout = DROPOUT
).to(device)

# Print the number of parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'The model has {n_params} parameters')

# Loss function
criterion = torch.nn.MSELoss()
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Train the model
train_res = utils.train(
    train_loader = dataloader_train,
    val_loader = dataloader_val,
    model = model,
    optimizer = optimizer,
    criterion = criterion,
    device = device,
    n_epochs = 3,
)

# Plot loss evolution
plt.figure(figsize=(10, 5))
plt.plot(train_res['losses_train'], label='Train loss')
plt.plot(train_res['losses_val'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
evaluation_res = utils.evaluate(
    model = model,
    loader = dataloader_test,
    criterion = criterion,
    device = device
)

true = np.concatenate(evaluation_res['true'], axis=0)
pred = np.concatenate(evaluation_res['pred'], axis=0)

print(f'sMAPE: {utils.sMAPE(true,pred):.1f}%')

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(true)), y=true, mode='lines', name='True'))
fig.add_trace(go.Scatter(x=np.arange(len(pred)), y=pred, mode='lines', name='Predicted'))
fig.show()

### Joel's model

In [ ]:

# Model parameters
INPUT_SIZE = X_train.shape[1]
HIDDEN_SIZE = 64
NUM_LAYERS = 2
OUTPUT_SIZE = y_train.shape[1]
DROPOUT = 0.0
# Optimizer parameters
LEARNING_RATE = 1e-3

# Build the model
model = LSTM.LSTM1(
    input_size = INPUT_SIZE,
    output_size = OUTPUT_SIZE,
    hidden_size = HIDDEN_SIZE,
    num_layers = NUM_LAYERS,
    dropout = DROPOUT
).to(device)

# Print the number of parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'The model has {n_params} parameters')

# Loss function
criterion = torch.nn.MSELoss()
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Train the model
train_res = utils.train(
    train_loader = dataloader_train,
    val_loader = dataloader_val,
    model = model,
    optimizer = optimizer,
    criterion = criterion,
    device = device,
    n_epochs = 3,
)

# Plot loss evolution
plt.figure(figsize=(10, 5))
plt.plot(train_res['losses_train'], label='Train loss')
plt.plot(train_res['losses_val'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
evaluation_res = utils.evaluate(
    model = model,
    loader = dataloader_test,
    criterion = criterion,
    device = device
)

true = np.concatenate(evaluation_res['true'], axis=0)
pred = np.concatenate(evaluation_res['pred'], axis=0)

In [ ]:
print(f'sMAPE: {utils.sMAPE(true,pred):.1f}%')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(true)), y=true, mode='lines', name='True'))
fig.add_trace(go.Scatter(x=np.arange(len(pred)), y=pred, mode='lines', name='Predicted'))
fig.show()

### Astice

<img src="../astice.png" width="200" >

In [ ]:
# Model parameters
INPUT_SIZE = X_train.shape[1]
OUTPUT_SIZE = y_train.shape[1]
HIDDEN_SIZE = 64

# LSTM
LSTM_SIZE = 64
LSTM_LAYERS = 2

# CMM
SMOOTH_CONV_CHANNELS = 4

# Optimizer parameters
LEARNING_RATE = 1e-3

# Build the model
model = LSTM.Astice(
    input_size = INPUT_SIZE,
    output_size = OUTPUT_SIZE,
    hidden_size = HIDDEN_SIZE,
    lstm_size = LSTM_SIZE,
    lstm_layers = LSTM_LAYERS,
    smooth_conv_channels = SMOOTH_CONV_CHANNELS
).to(device)

# Print the number of parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'The model has {n_params} parameters')

# Loss function
criterion = torch.nn.MSELoss()
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Train the model
train_res = utils.train(
    train_loader = dataloader_train,
    val_loader = dataloader_val,
    model = model,
    optimizer = optimizer,
    criterion = criterion,
    device = device,
    n_epochs = 3,
)

# Plot loss evolution
plt.figure(figsize=(10, 5))
plt.plot(train_res['losses_train'], label='Train loss')
plt.plot(train_res['losses_val'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
evaluation_res = utils.evaluate(
    model = model,
    loader = dataloader_test,
    criterion = criterion,
    device = device
)

true = np.concatenate(evaluation_res['true'], axis=0)
pred = np.concatenate(evaluation_res['pred'], axis=0)

In [ ]:
print(f'sMAPE: {utils.sMAPE(true,pred):.1f}%')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(true)), y=true, mode='lines', name='True'))
fig.add_trace(go.Scatter(x=np.arange(len(pred)), y=pred, mode='lines', name='Predicted'))
fig.show()